# LeetCode 1207 — Unique Number of Occurrences bằng Python

**Link đề:** https://leetcode.com/problems/unique-number-of-occurrences/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table

```python
class Solution:
    def uniqueOccurrences(self, arr: list[int]) -> bool:
        ...
```

## Mục tiêu học
1. Thành thạo **hai tầng đếm**: đếm *giá trị* → được *số lần xuất hiện*; rồi kiểm tra các số lần đó có **trùng nhau** không.
2. Nắm `dict`, `Counter`, `.values()`, `set`, so sánh `len(set(x)) == len(x)` để phát hiện trùng.
3. Đi từ cách dùng `arr.count` (`O(n²)`) → hash map + set (`O(n)`) → sort + đếm đoạn → mảng đếm có offset.
4. Trace từng bước, chứng minh, test đối chiếu và làm bài tập có đáp án cuối notebook.

## 1. Đề bài và ví dụ

Cho mảng số nguyên `arr`. Trả `True` nếu **số lần xuất hiện của mỗi giá trị là duy nhất** (không có hai giá trị nào xuất hiện cùng số lần), ngược lại trả `False`.

| `arr` | Đếm | Kết quả |
|---|---|:---:|
| `[1, 2, 2, 1, 1, 3]` | 1→3, 2→2, 3→1 | `True` (3, 2, 1 khác nhau) |
| `[1, 2]` | 1→1, 2→1 | `False` (hai giá trị cùng xuất hiện 1 lần) |
| `[-3, 0, 1, -3, 1, 1, 1, -3, 10, 0]` | -3→3, 0→2, 1→4, 10→1 | `True` |

**Constraints:** `1 <= arr.length <= 1000`; `-1000 <= arr[i] <= 1000`.

Hãy để ý: câu hỏi **không** phải "các giá trị có trùng nhau không" (chắc chắn có), mà là "các **số đếm** có trùng nhau không".

## 2. Nền tảng Python cần biết

### 2.1. `dict` đếm tần suất với `get`

In [1]:
arr = [1, 2, 2, 1, 1, 3]
freq = {}
for x in arr:
    freq[x] = freq.get(x, 0) + 1
print("freq =", freq)
print("keys   :", list(freq.keys()))
print("values :", list(freq.values()))
print("items  :", list(freq.items()))

freq = {1: 3, 2: 2, 3: 1}
keys   : [1, 2, 3]
values : [3, 2, 1]
items  : [(1, 3), (2, 2), (3, 1)]


- `.keys()` — các giá trị đã gặp (1, 2, 3).
- `.values()` — **số lần xuất hiện** (3, 2, 1). Đây chính là thứ ta cần kiểm tra trùng.
- `.items()` — từng cặp `(giá trị, số lần)`.

### 2.2. `collections.Counter`

In [2]:
from collections import Counter
c = Counter([-3, 0, 1, -3, 1, 1, 1, -3, 10, 0])
print(c)
print("số lần của 1:", c[1], "| của 99:", c[99])
print("most_common(2):", c.most_common(2))

Counter({1: 4, -3: 3, 0: 2, 10: 1})
số lần của 1: 4 | của 99: 0
most_common(2): [(1, 4), (-3, 3)]


### 2.3. `set` — tập hợp không chứa phần tử trùng

`set(iterable)` bỏ trùng. Nếu `len(set(x)) < len(x)` thì `x` **có phần tử lặp**. Thao tác `in`, `add` trung bình `O(1)`.

In [3]:
counts = [3, 2, 1]
print(set(counts), len(set(counts)) == len(counts))
counts = [1, 1]
print(set(counts), len(set(counts)) == len(counts))
seen = set()
seen.add(5); seen.add(5); seen.add(7)
print("seen =", seen, "| 5 in seen:", 5 in seen, "| 6 in seen:", 6 in seen)

{1, 2, 3} True
{1} False
seen = {5, 7} | 5 in seen: True | 6 in seen: False


### 2.4. `list.count(x)` — đếm trực tiếp nhưng tốn `O(n)` mỗi lần

In [4]:
arr = [1, 2, 2, 1, 1, 3]
print("arr.count(1) =", arr.count(1), "| arr.count(4) =", arr.count(4))

arr.count(1) = 3 | arr.count(4) = 0


## 3. Cách 1 — Trực giác: dùng `count` cho từng giá trị khác nhau

1. Lấy các giá trị khác nhau: `set(arr)`.
2. Với mỗi giá trị, đếm `arr.count(v)` → danh sách số lần.
3. So từng cặp số lần xem có bằng nhau không.

In [5]:
def unique_occ_naive(arr: list[int]) -> bool:
    values = list(set(arr))
    counts = [arr.count(v) for v in values]        # mỗi lần count là O(n)
    for i in range(len(counts)):
        for j in range(i + 1, len(counts)):
            if counts[i] == counts[j]:
                return False
    return True

for arr in [[1, 2, 2, 1, 1, 3], [1, 2], [-3, 0, 1, -3, 1, 1, 1, -3, 10, 0]]:
    print(arr, "->", unique_occ_naive(arr))

[1, 2, 2, 1, 1, 3] -> True
[1, 2] -> False
[-3, 0, 1, -3, 1, 1, 1, -3, 10, 0] -> True


**Chi phí:** gọi `count` cho `k` giá trị khác nhau → `O(k·n)`; so từng cặp số lần → `O(k²)`. Tệ nhất `k ≈ n` → `O(n²)`.
Với `n ≤ 1000` vẫn chạy được, nhưng đây là thói quen xấu khi `n` lớn.

## 4. Nút thắt và insight

- **Nút thắt 1:** `arr.count(v)` duyệt lại cả mảng cho **mỗi** giá trị. → Đếm **một lần** bằng hash map.
- **Nút thắt 2:** so từng cặp số lần là `O(k²)`. → "Có phần tử trùng không?" là câu hỏi kinh điển của **set**: hoặc `len(set) == len(list)`, hoặc duyệt và kiểm tra `in seen`.

Bài toán = **Contains Duplicate** áp lên danh sách `freq.values()`.

In [6]:
from collections import Counter

arr = [1, 2, 2, 1, 1, 3]
freq = Counter(arr)
counts = list(freq.values())
print("counts =", counts, "| set =", set(counts))
print("Không trùng?", len(set(counts)) == len(counts))

counts = [3, 2, 1] | set = {1, 2, 3}
Không trùng? True


## 5. Lời giải chính — giảng từng đoạn rồi ghép

### Cách 2 — Counter + so sánh độ dài set

**Đoạn 1:** `freq = Counter(arr)` — tầng đếm thứ nhất.
**Đoạn 2:** `counts = freq.values()` — các số lần.
**Đoạn 3:** `len(set(counts)) == len(counts)` — kiểm tra trùng.

In [7]:
from collections import Counter

def unique_occ_set(arr: list[int]) -> bool:
    freq = Counter(arr)                         # đoạn 1
    counts = freq.values()                      # đoạn 2
    return len(set(counts)) == len(counts)      # đoạn 3

for arr in [[1, 2, 2, 1, 1, 3], [1, 2], [-3, 0, 1, -3, 1, 1, 1, -3, 10, 0]]:
    print(arr, "->", unique_occ_set(arr))

[1, 2, 2, 1, 1, 3] -> True
[1, 2] -> False
[-3, 0, 1, -3, 1, 1, 1, -3, 10, 0] -> True


### Cách 3 — Dict thủ công + `seen` có dừng sớm

Không dùng thư viện; dừng ngay khi thấy một số lần đã xuất hiện:

In [8]:
def unique_occ_seen(arr: list[int]) -> bool:
    freq = {}
    for x in arr:
        freq[x] = freq.get(x, 0) + 1
    seen = set()
    for c in freq.values():
        if c in seen:
            return False          # hai giá trị có cùng số lần
        seen.add(c)
    return True

for arr in [[1, 2, 2, 1, 1, 3], [1, 2], [-3, 0, 1, -3, 1, 1, 1, -3, 10, 0]]:
    print(arr, "->", unique_occ_seen(arr))

[1, 2, 2, 1, 1, 3] -> True
[1, 2] -> False
[-3, 0, 1, -3, 1, 1, 1, -3, 10, 0] -> True


### Cách 4 — Sắp xếp rồi đếm các đoạn liên tiếp

Sau khi sort, các giá trị bằng nhau đứng **liền nhau** → độ dài mỗi đoạn là số lần xuất hiện. Không cần hash map để đếm.

In [9]:
def unique_occ_sort(arr: list[int]) -> bool:
    s = sorted(arr)
    runs = []
    run = 1
    for i in range(1, len(s)):
        if s[i] == s[i - 1]:
            run += 1
        else:
            runs.append(run)
            run = 1
    runs.append(run)                      # đừng quên đoạn cuối cùng
    runs.sort()
    return all(runs[i] != runs[i - 1] for i in range(1, len(runs)))

print(sorted([1, 2, 2, 1, 1, 3]))
for arr in [[1, 2, 2, 1, 1, 3], [1, 2], [-3, 0, 1, -3, 1, 1, 1, -3, 10, 0]]:
    print(arr, "->", unique_occ_sort(arr))

[1, 1, 1, 2, 2, 3]
[1, 2, 2, 1, 1, 3] -> True
[1, 2] -> False
[-3, 0, 1, -3, 1, 1, 1, -3, 10, 0] -> True


Ở bước cuối ta **sort lại các độ dài** rồi kiểm tra kề nhau khác nhau — chính là kỹ thuật "sort để phát hiện trùng".

### Cách 5 — Mảng đếm có offset (vì `-1000 <= arr[i] <= 1000`)

Chỉ số list không âm, nên dời giá trị: `idx = x + 1000` ∈ `[0, 2000]`. Số lần xuất hiện tối đa là `n ≤ 1000` → dùng thêm mảng `used` kích thước `n + 1` để đánh dấu số lần đã thấy.

In [10]:
def unique_occ_array(arr: list[int]) -> bool:
    OFFSET = 1000
    cnt = [0] * 2001
    for x in arr:
        cnt[x + OFFSET] += 1
    used = [False] * (len(arr) + 1)
    for c in cnt:
        if c == 0:
            continue                  # giá trị không xuất hiện
        if used[c]:
            return False
        used[c] = True
    return True

print("-1000 + 1000 =", -1000 + 1000, "| 1000 + 1000 =", 1000 + 1000)
for arr in [[1, 2, 2, 1, 1, 3], [1, 2], [-3, 0, 1, -3, 1, 1, 1, -3, 10, 0]]:
    print(arr, "->", unique_occ_array(arr))

-1000 + 1000 = 0 | 1000 + 1000 = 2000
[1, 2, 2, 1, 1, 3] -> True
[1, 2] -> False
[-3, 0, 1, -3, 1, 1, 1, -3, 10, 0] -> True


## 6. TRACE từng bước

Ví dụ 3: `arr = [-3, 0, 1, -3, 1, 1, 1, -3, 10, 0]`.

**Pha 1 — đếm:**

| i | x | freq sau bước |
|:---:|:---:|---|
| 0 | -3 | {-3:1} |
| 1 | 0 | {-3:1, 0:1} |
| 2 | 1 | {-3:1, 0:1, 1:1} |
| ... | ... | ... |
| 9 | 0 | {-3:3, 0:2, 1:4, 10:1} |

**Pha 2 — kiểm tra số lần:**

| giá trị | số lần c | seen trước | c in seen? | Hành động |
|:---:|:---:|---|:---:|---|
| -3 | 3 | {} | không | thêm 3 |
| 0 | 2 | {3} | không | thêm 2 |
| 1 | 4 | {2,3} | không | thêm 4 |
| 10 | 1 | {2,3,4} | không | thêm 1 → `True` |

In [11]:
def trace_unique_occ(arr: list[int]) -> bool:
    freq = {}
    print("Pha 1 — đếm")
    for i, x in enumerate(arr):
        freq[x] = freq.get(x, 0) + 1
        print(f"  i={i} x={x:3d} -> freq={freq}")
    print("Pha 2 — kiểm tra số lần")
    seen = set()
    for value, c in freq.items():
        if c in seen:
            print(f"  giá trị {value}: c={c} đã có trong seen={seen} -> False")
            return False
        seen.add(c)
        print(f"  giá trị {value}: c={c} mới -> seen={seen}")
    print("  Không trùng -> True")
    return True

assert trace_unique_occ([-3, 0, 1, -3, 1, 1, 1, -3, 10, 0]) is True

Pha 1 — đếm
  i=0 x= -3 -> freq={-3: 1}
  i=1 x=  0 -> freq={-3: 1, 0: 1}
  i=2 x=  1 -> freq={-3: 1, 0: 1, 1: 1}
  i=3 x= -3 -> freq={-3: 2, 0: 1, 1: 1}
  i=4 x=  1 -> freq={-3: 2, 0: 1, 1: 2}
  i=5 x=  1 -> freq={-3: 2, 0: 1, 1: 3}
  i=6 x=  1 -> freq={-3: 2, 0: 1, 1: 4}
  i=7 x= -3 -> freq={-3: 3, 0: 1, 1: 4}
  i=8 x= 10 -> freq={-3: 3, 0: 1, 1: 4, 10: 1}
  i=9 x=  0 -> freq={-3: 3, 0: 2, 1: 4, 10: 1}
Pha 2 — kiểm tra số lần
  giá trị -3: c=3 mới -> seen={3}
  giá trị 0: c=2 mới -> seen={2, 3}
  giá trị 1: c=4 mới -> seen={2, 3, 4}
  giá trị 10: c=1 mới -> seen={1, 2, 3, 4}
  Không trùng -> True


Trace một ca trả `False` để thấy nhánh dừng sớm:

In [12]:
assert trace_unique_occ([5, 7, 7, 5, 9]) is False

Pha 1 — đếm
  i=0 x=  5 -> freq={5: 1}
  i=1 x=  7 -> freq={5: 1, 7: 1}
  i=2 x=  7 -> freq={5: 1, 7: 2}
  i=3 x=  5 -> freq={5: 2, 7: 2}
  i=4 x=  9 -> freq={5: 2, 7: 2, 9: 1}
Pha 2 — kiểm tra số lần
  giá trị 5: c=2 mới -> seen={2}
  giá trị 7: c=2 đã có trong seen={2} -> False


## 7. Code nộp LeetCode

### 7.1. Bản chính — Counter + set

In [13]:
from typing import List
from collections import Counter

class Solution:
    def uniqueOccurrences(self, arr: List[int]) -> bool:
        counts = Counter(arr).values()
        return len(counts) == len(set(counts))

sol = Solution()
print(sol.uniqueOccurrences([1, 2, 2, 1, 1, 3]), sol.uniqueOccurrences([1, 2]), sol.uniqueOccurrences([-3, 0, 1, -3, 1, 1, 1, -3, 10, 0]))

True False True


### 7.2. Bản không thư viện, dừng sớm

In [14]:
class SolutionSeen:
    def uniqueOccurrences(self, arr: List[int]) -> bool:
        freq = {}
        for x in arr:
            freq[x] = freq.get(x, 0) + 1
        seen = set()
        for c in freq.values():
            if c in seen:
                return False
            seen.add(c)
        return True

assert SolutionSeen().uniqueOccurrences([1, 2]) is False
print("SolutionSeen OK")

SolutionSeen OK


### 7.3. Bản sort + đếm đoạn

In [15]:
class SolutionSort:
    def uniqueOccurrences(self, arr: List[int]) -> bool:
        arr = sorted(arr)
        runs, run = [], 1
        for i in range(1, len(arr)):
            if arr[i] == arr[i - 1]:
                run += 1
            else:
                runs.append(run)
                run = 1
        runs.append(run)
        return len(runs) == len(set(runs))

assert SolutionSort().uniqueOccurrences([1, 2, 2, 1, 1, 3]) is True
print("SolutionSort OK")

SolutionSort OK


## 8. Vì sao đúng?

- Sau pha đếm, `freq[v]` = **đúng** số lần `v` xuất hiện (mỗi phần tử cộng đúng 1 vào đúng ô của nó). Bất biến: sau khi xử lý `i` phần tử đầu, `freq` là bảng đếm của `arr[0..i-1]`.
- Đề hỏi: "có hai giá trị **khác nhau** `u ≠ v` mà `freq[u] == freq[v]` không?". Mỗi giá trị khác nhau chiếm đúng một phần tử trong `freq.values()`. Vậy câu hỏi ⇔ "`freq.values()` có phần tử lặp không?".
- `set` giữ mỗi giá trị một lần, nên `len(set(counts)) == len(counts)` ⇔ không có số lần nào lặp. ∎

Với cách sort: các giá trị bằng nhau đứng liền nhau, nên mỗi "đoạn" ứng với đúng một giá trị và độ dài đoạn = số lần xuất hiện.

## 9. Độ phức tạp và so sánh

Gọi `n = len(arr)`, `k` = số giá trị khác nhau (`k ≤ n`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| `count` + so cặp | `O(k·n + k²)` ⊂ `O(n²)` | `O(k)` | dễ nghĩ, chậm |
| Counter + set | `O(n)` trung bình | `O(k)` | ngắn nhất |
| Dict + seen (dừng sớm) | `O(n)` trung bình | `O(k)` | không thư viện |
| Sort + đếm đoạn | `O(n log n)` | `O(n)` | không cần hash |
| Mảng đếm offset | `O(n + R)`, `R = 2001` | `O(R + n)` | tận dụng miền nhỏ |

"Trung bình" vì thao tác hash (`dict`, `set`) là `O(1)` trung bình, không phải tuyệt đối.

## 10. Bộ test

Mẫu đề, 1 phần tử, tất cả giống nhau, nhiều giá trị cùng xuất hiện 1 lần, số âm/biên, và đối chiếu mọi cách trên dữ liệu ngẫu nhiên.

In [16]:
impls = {
    "naive": unique_occ_naive,
    "set": unique_occ_set,
    "seen": unique_occ_seen,
    "sort": unique_occ_sort,
    "array": unique_occ_array,
    "Solution": Solution().uniqueOccurrences,
    "SolSeen": SolutionSeen().uniqueOccurrences,
    "SolSort": SolutionSort().uniqueOccurrences,
}
cases = [
    ([1, 2, 2, 1, 1, 3], True),
    ([1, 2], False),
    ([-3, 0, 1, -3, 1, 1, 1, -3, 10, 0], True),
    ([7], True),                          # 1 phần tử
    ([4, 4, 4, 4], True),                 # 1 giá trị duy nhất
    ([1, 2, 3], False),                   # mọi giá trị 1 lần
    ([-1000, 1000, 1000], True),          # biên giá trị
    ([1, 1, 2, 2, 3], False),             # 1 và 2 cùng 2 lần
    ([5, 7, 7, 5, 9], False),
]
for arr, expected in cases:
    for name, f in impls.items():
        assert f(arr) == expected, (name, arr)
    print(f"{str(arr):40} -> {expected} PASS")

[1, 2, 2, 1, 1, 3]                       -> True PASS
[1, 2]                                   -> False PASS
[-3, 0, 1, -3, 1, 1, 1, -3, 10, 0]       -> True PASS
[7]                                      -> True PASS
[4, 4, 4, 4]                             -> True PASS
[1, 2, 3]                                -> False PASS
[-1000, 1000, 1000]                      -> True PASS
[1, 1, 2, 2, 3]                          -> False PASS
[5, 7, 7, 5, 9]                          -> False PASS


In [17]:
import random

rng = random.Random(1207)
for _ in range(500):
    arr = [rng.randint(-5, 5) for _ in range(rng.randint(1, 30))]
    res = {name: f(arr) for name, f in impls.items()}
    assert len(set(res.values())) == 1, (arr, res)
big = [v for v in range(-1000, -956) for _ in range(v + 1001)]   # các số lần 1..44 khác nhau
print("len(big) =", len(big), "->", Solution().uniqueOccurrences(big))
assert Solution().uniqueOccurrences(big) is True
print("500 test ngẫu nhiên: mọi cách khớp nhau")

len(big) = 990 -> True
500 test ngẫu nhiên: mọi cách khớp nhau


## 11. Lỗi người mới hay gặp

### Lỗi 1: Kiểm tra trùng trên **giá trị** thay vì trên **số lần**

In [18]:
def wrong_check_values(arr):
    return len(set(arr)) == len(arr)     # SAI: đây là Contains Duplicate trên arr

print("sai :", wrong_check_values([1, 2, 2, 1, 1, 3]), "| đúng:", unique_occ_set([1, 2, 2, 1, 1, 3]))

sai : False | đúng: True


Vì sao sai: `arr` có phần tử lặp nên trả `False`, nhưng số lần (3, 2, 1) đều khác nhau → đáp án `True`. Sửa: áp `set` lên `freq.values()`.

### Lỗi 2: So `len(set(counts))` với `len(arr)`

In [19]:
from collections import Counter

def wrong_compare_len(arr):
    counts = Counter(arr).values()
    return len(set(counts)) == len(arr)     # SAI: phải so với len(counts)

print("sai :", wrong_compare_len([1, 2, 2, 1, 1, 3]), "| đúng:", unique_occ_set([1, 2, 2, 1, 1, 3]))

sai : False | đúng: True


### Lỗi 3: Quên đoạn cuối khi đếm theo đoạn sau sort

In [20]:
def wrong_forget_last_run(arr):
    s = sorted(arr)
    runs, run = [], 1
    for i in range(1, len(s)):
        if s[i] == s[i - 1]:
            run += 1
        else:
            runs.append(run)
            run = 1
    # SAI: thiếu runs.append(run)
    return len(runs) == len(set(runs))

print("sai :", wrong_forget_last_run([1, 2, 3, 3]), "| đúng:", unique_occ_sort([1, 2, 3, 3]))

sai : False | đúng: False


Với `[1, 2, 3, 3]` (số lần 1, 1, 2 → `False`), bản sai chỉ thấy `[1, 1]` nên **tình cờ** vẫn ra `False`. Test "may mắn" như vậy rất nguy hiểm — hãy xem một ca làm lộ lỗi:

In [21]:
print("sai :", wrong_forget_last_run([1, 2, 2, 3]), "| đúng:", unique_occ_sort([1, 2, 2, 3]))

sai : True | đúng: False


`[1, 2, 2, 3]` có số lần 1, 2, 1 → `False`, nhưng bản sai bỏ mất đoạn `3` (độ dài 1) nên chỉ thấy `[1, 2]` → `True`. Sửa: luôn `append` đoạn cuối sau vòng lặp.

### Lỗi 4: Dùng chỉ số âm với mảng đếm mà quên offset

`cnt[x] += 1` với `x = -3` không báo lỗi trong Python (chỉ số âm đếm từ cuối!) → ghi nhầm ô một cách im lặng. Luôn cộng `OFFSET`.

In [22]:
cnt = [0] * 5
cnt[-1] += 1          # không lỗi! nhưng thực ra tăng ô cuối cùng cnt[4]
print(cnt)

[0, 0, 0, 0, 1]


## 12. Mẹo nhận dạng pattern

- **"Số lần xuất hiện"**, **"tần suất"** → bước 1 luôn là `Counter` / `dict`.
- **"Duy nhất / có trùng không"** → `set` (so độ dài hoặc `in seen`).
- Bài 1207 = *đếm* rồi *Contains Duplicate trên các số đếm* — kiểu ghép 2 kỹ thuật cơ bản rất phổ biến ở Easy.

Bài cùng họ: 217 Contains Duplicate, 1636 Sort Array by Increasing Frequency, 451 Sort Characters By Frequency, 1647 Minimum Deletions to Make Character Frequencies Unique (biến thể Medium — làm số lần thành duy nhất).

## 13. Biến thể

### 13.1. Trả về các số lần bị trùng

In [23]:
from collections import Counter

def duplicated_counts(arr):
    freq_of_freq = Counter(Counter(arr).values())     # đếm hai tầng!
    return sorted(c for c, k in freq_of_freq.items() if k > 1)

print(duplicated_counts([1, 2, 3, 3, 4, 4, 5]))   # 1 (ba giá trị 1,2,5) và 2 (hai giá trị 3,4)

[1, 2]


`Counter(Counter(arr).values())` = "có bao nhiêu giá trị xuất hiện đúng `c` lần" — đếm của đếm.

### 13.2. Minimum Deletions to Make Character Frequencies Unique (LeetCode 1647)

Xoá ít ký tự nhất để mọi số lần (khác 0) là duy nhất. Greedy: duyệt số lần giảm dần, nếu số lần đã bị dùng thì giảm dần tới khi gặp số chưa dùng (hoặc 0).

In [24]:
def min_deletions(s: str) -> int:
    used = set()
    deletions = 0
    for c in sorted(Counter(s).values(), reverse=True):
        while c > 0 and c in used:
            c -= 1
            deletions += 1
        if c > 0:
            used.add(c)
    return deletions

print(min_deletions("aab"), min_deletions("aaabbbcc"), min_deletions("ceabaacb"))   # 0 2 2

0 2 2


### 13.3. Kiểm tra bằng một dòng với `Counter` hai tầng

Số lần là duy nhất ⇔ mọi giá trị trong `Counter(counts)` đều bằng 1.

In [25]:
def unique_occ_two_level(arr):
    return all(k == 1 for k in Counter(Counter(arr).values()).values())

print(unique_occ_two_level([1, 2, 2, 1, 1, 3]), unique_occ_two_level([1, 2]))

True False


## 14. Bài tập (TODO — không gọi khi chạy)

**Bài 1 (tính tay).** `arr = [2, 2, 3, 3, 3, 4, 4, 4, 4, 5]`. Lập bảng đếm và cho kết quả.

**Bài 2 (dự đoán output).** `len(set([3, 2, 3]))`, `Counter("aab").values()` chứa những số nào, `list({1: 5, 2: 5}.values())`.

**Bài 3 (điền TODO).** Hoàn thiện đếm bằng dict và kiểm tra bằng set.

In [26]:
def exercise_unique_occ(arr: list[int]) -> bool:
    freq = {}
    # TODO 1: đếm từng x
    # TODO 2: trả True nếu các số lần không trùng nhau
    pass

# assert exercise_unique_occ([1, 2, 2, 1, 1, 3]) is True
# assert exercise_unique_occ([1, 2]) is False

**Bài 4 (viết lại).** Viết lời giải dùng **sort + đếm đoạn**, không dùng `dict`/`Counter`/`set` (dùng sort lần hai cho danh sách độ dài).

In [27]:
def exercise_unique_occ_sort(arr: list[int]) -> bool:
    # TODO
    pass

# assert exercise_unique_occ_sort([-3, 0, 1, -3, 1, 1, 1, -3, 10, 0]) is True
# assert exercise_unique_occ_sort([1, 2, 2, 3]) is False

**Bài 5 (biến thể).** Cho chuỗi `s`, trả `True` nếu **mọi ký tự xuất hiện cùng số lần** (LeetCode 1941 — Check if All Characters Have Equal Number of Occurrences).

In [28]:
def exercise_equal_occ(s: str) -> bool:
    # TODO
    pass

# assert exercise_equal_occ("abacbc") is True
# assert exercise_equal_occ("aaabb") is False

**Bài 6 (mini challenge).** Trả về **số phần tử ít nhất phải xoá** khỏi `arr` để các số lần xuất hiện (khác 0) đều duy nhất (giống 1647 nhưng với mảng số).

In [29]:
def exercise_min_deletions_arr(arr: list[int]) -> int:
    # TODO
    pass

# assert exercise_min_deletions_arr([1, 2]) == 1
# assert exercise_min_deletions_arr([1, 1, 2, 2, 3, 3]) == 3

## 15. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Đếm từng giá trị | So các số lần | Có hai số lần nào bằng nhau không? |
| 2 | set bỏ trùng | Counter đếm ký tự | values có thể lặp |
| 3 | freq.get | `counts = list(freq.values())` | `len(set(counts)) == len(counts)` |
| 4 | Giá trị bằng nhau nằm liền nhau | `run` và `runs` | sort `runs`, so kề nhau; đừng quên đoạn cuối |
| 5 | Ngược lại bài gốc | set của số lần | `len(set(Counter(s).values())) == 1` |
| 6 | Xét số lần giảm dần | `used` set | `while c > 0 and c in used: c -= 1; deletions += 1` |

## 16. Đáp án tham khảo

**Bài 1:** 2→2, 3→3, 4→4, 5→1 → số lần 2, 3, 4, 1 khác nhau → `True`.

**Bài 2:** `2`; `2` và `1`; `[5, 5]`.

In [30]:
from collections import Counter

def answer_unique_occ(arr):
    freq = {}
    for x in arr:
        freq[x] = freq.get(x, 0) + 1
    counts = list(freq.values())
    return len(set(counts)) == len(counts)

def answer_unique_occ_sort(arr):
    s = sorted(arr)
    runs, run = [], 1
    for i in range(1, len(s)):
        if s[i] == s[i - 1]:
            run += 1
        else:
            runs.append(run)
            run = 1
    runs.append(run)
    runs.sort()
    for i in range(1, len(runs)):
        if runs[i] == runs[i - 1]:
            return False
    return True

def answer_equal_occ(s):
    return len(set(Counter(s).values())) == 1

def answer_min_deletions_arr(arr):
    used, deletions = set(), 0
    for c in sorted(Counter(arr).values(), reverse=True):
        while c > 0 and c in used:
            c -= 1
            deletions += 1
        if c > 0:
            used.add(c)
    return deletions

assert unique_occ_set([2, 2, 3, 3, 3, 4, 4, 4, 4, 5]) is True
assert len(set([3, 2, 3])) == 2 and sorted(Counter("aab").values()) == [1, 2] and list({1: 5, 2: 5}.values()) == [5, 5]
assert answer_unique_occ([1, 2, 2, 1, 1, 3]) is True and answer_unique_occ([1, 2]) is False
assert answer_unique_occ_sort([-3, 0, 1, -3, 1, 1, 1, -3, 10, 0]) is True
assert answer_unique_occ_sort([1, 2, 2, 3]) is False
assert answer_equal_occ("abacbc") is True and answer_equal_occ("aaabb") is False
assert answer_min_deletions_arr([1, 2]) == 1
assert answer_min_deletions_arr([1, 1, 2, 2, 3, 3]) == 3
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


## 17. Template nhớ nhanh và checklist submit

```python
from collections import Counter

counts = Counter(arr).values()          # tầng 1: đếm giá trị
return len(counts) == len(set(counts))  # tầng 2: số lần có trùng không?
```

**Câu thần chú:** *"Đếm trước, rồi hỏi các con số đếm có trùng không."*

### Checklist trước khi submit
- [ ] Áp `set` lên **số lần** (`values()`), không phải lên `arr`?
- [ ] So `len(set(counts))` với `len(counts)` (không phải `len(arr)`)?
- [ ] Nếu đếm theo đoạn sau sort: đã thêm **đoạn cuối**?
- [ ] Nếu dùng mảng đếm: đã cộng **offset** cho số âm?
- [ ] Tên method `uniqueOccurrences`, trả `bool`?